#### 1.数据初始化及定义

In [1]:
import json
import copy
from torch.utils.data import Dataset
import datas
class RaftLogDataSet(Dataset):
    def __init__(self, datafilesName):
        self.datafilesName = datafilesName
        self.serversOrignDatas = []
        for fileName in datafilesName:
           with open(fileName, 'r') as f:
                lines = f.readlines()
                tmp = []
                for line in lines:
                    if len(line) < 2:continue
                    js_data = json.loads(line)
                    tmp.append(js_data)
                self.serversOrignDatas.append(tmp)
                f.close()
        print("load datafiles done...")      
        self.all_servers_datasets = []
        
       
        for serverOrignData in self.serversOrignDatas:
            flattend_jsons = []
            for js_data in serverOrignData:
                if ('role' not in js_data) or (line is None) or ('timestamp' not in js_data):
                    continue
                keys,values = datas.flattenJson(js_data)
                if(len(keys) != len(values)):continue
                tmp_js = dict(zip(keys,values))
                if(tmp_js['role'] == 'state'):
                    tmp_js.pop('raft_state.ip_index')
                    tmp_js.pop('raft_state.nextIndex')
                    lo_rx_str = 'system_state.network.process.lo:.RX Bytes'
                    lo_tx_str = 'system_state.network.process.lo:.TX Bytes'
                    tmp_js.pop(lo_rx_str)
                    tmp_js.pop(lo_tx_str)
                flattend_jsons.append(tmp_js)
            self.all_servers_datasets.append(flattend_jsons)
        print("flattend_jsons ...")
        combined_actionTuples = []
        '''
        combined_actionTuples :[ (timestamp,[(serverId,actionJson,index) ... ]) ... ]
        '''
        def timestampIsExistInTuples(combined_actionTuples,timeStamp) -> int:
            for i in range(len(combined_actionTuples)):
                if combined_actionTuples[i][0] == timeStamp:
                    return i
            return -1
        
        def getPrevStateJsonInServerByIndex(serverId,actionIndex):
            serverDatas = self.all_servers_datasets[serverId]
            for i in range(actionIndex,-1,-1):
                if serverDatas[i]['role'] == 'state':
                    return serverDatas[i]
            return None

        def getNextStateJsonInServerByIndex(serverId,actionIndex):
            serverDatas = self.all_servers_datasets[serverId]
            for i in range(actionIndex,len(serverDatas)):
                if serverDatas[i]['role'] == 'state':
                    return serverDatas[i]
            return None
        
        for serverId in range(len(self.all_servers_datasets)):
            serverDatas = self.all_servers_datasets[serverId]
            for i  in range(len(serverDatas)):
                json_data = serverDatas[i]
                if(json_data['role'] != 'action'):continue
                timestamp = json_data['timestamp']
                idx = timestampIsExistInTuples(combined_actionTuples,timestamp)
                tuple_ = (serverId,json_data,i)
                if idx == -1:
                    combined_actionTuples.append((timestamp,[tuple_]))
                else:   
                    combined_actionTuples[idx][1].append(tuple_)
                    
        self.train_data_each_batch_by_logicOrder = []
        for tp_ in combined_actionTuples:
            # tp_ : (timestamp,[(serverId,actionJson,index) ... ])
            actionTupleList = tp_[-1]
            grouped_by_serverId = {}
            for actionTuple in actionTupleList:
                # actionTuple : (serverId,actionJson,index)
                serverId = actionTuple[0]
                if serverId not in grouped_by_serverId:
                    grouped_by_serverId[serverId] = []
                grouped_by_serverId[serverId].append(actionTuple)
                
            batch_tuple = {}
            for serverId in grouped_by_serverId:
                batch_tuple[serverId] = {
                    'prevStateJson' : None,
                    'actions' : [],
                    'nextStateJson':None
                }
                actionTuples = grouped_by_serverId[serverId]
                prevStateJson = None
                nextStateJson = None
                # actionTuple: (serverId,actionJson,index)
                for actionTuple in actionTuples:
                    actionJson = actionTuple[1]
                    index = actionTuple[2]
                    if prevStateJson is None:
                        prevStateJson = getPrevStateJsonInServerByIndex(serverId,index)
                    if nextStateJson is None:
                        nextStateJson = getNextStateJsonInServerByIndex(serverId,index)
                    batch_tuple[serverId]['actions'].append(actionJson)
                batch_tuple[serverId]['prevStateJson'] = prevStateJson
                batch_tuple[serverId]['nextStateJson'] = nextStateJson
            self.train_data_each_batch_by_logicOrder.append(batch_tuple)
          

    def __getitem__(self, index):
        
        if(index >= len(self.train_data_each_batch_by_logicOrder) or index < 0):
            raise IndexError("Index out of range")
        batch_tuple = self.train_data_each_batch_by_logicOrder[index]
        once_prevStateJson = []
        once_actions = []
        once_nextStateJson = []
        for serverId in batch_tuple:
            item = batch_tuple[serverId]
            prevStateJson = item['prevStateJson']
            nextStateJson = item['nextStateJson']
            actions = item['actions']
            once_prevStateJson.append(prevStateJson)
            once_nextStateJson.append(nextStateJson)
            once_actions.append(actions)  
        return (once_prevStateJson,once_actions,once_nextStateJson  )
    
    def __len__(self):
        return len(self.train_data_each_batch_by_logicOrder)


In [2]:
datafilesName = ['/home/cdy/code/projects/cRaft/.data/system_data/server-128-0/system_runtime.data0',
                     '/home/cdy/code/projects/cRaft/.data/system_data/server-130-1/system_runtime.data0',
                     '/home/cdy/code/projects/cRaft/.data/system_data/server-131-2/system_runtime.data0',
                     '/home/cdy/code/projects/cRaft/.data/system_data/server-133-3/system_runtime.data0',
                     '/home/cdy/code/projects/cRaft/.data/system_data/server-134-4/system_runtime.data0',
                ]
# datafilesName = ['/home/cdy/code/projects/cRaft/src/train/data_util/tmpdata/system_data/server-128-0/system_runtime.data0',
#                      '/home/cdy/code/projects/cRaft/src/train/data_util/tmpdata/system_data/server-130-1/system_runtime.data0',
#                      '/home/cdy/code/projects/cRaft/src/train/data_util/tmpdata/system_data/server-131-2/system_runtime.data0',
#                      '/home/cdy/code/projects/cRaft/src/train/data_util/tmpdata/system_data/server-133-3/system_runtime.data0',
#                      '//home/cdy/code/projects/cRaft/src/train/data_util/tmpdata/system_data/server-134-4/system_runtime.data0',
#                 ]
dataset = RaftLogDataSet(datafilesName)


load datafiles done...
flattend_jsons ...


#### 2.构建网络

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class StateCriticNetwork(nn.Module):
    def __init__(self, input_channels, input_width, embed_dim, num_heads=8):
        super(StateCriticNetwork, self).__init__()
        self.input_channels = input_channels 
        self.input_width = input_width 
        self.embed_dim = embed_dim 
        self.num_heads = num_heads  
        
    
        self.conv1 = nn.Conv1d(in_channels=self.input_channels, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv1d(in_channels=64, out_channels=self.embed_dim, kernel_size=3, stride=1, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)  # 将序列长度压缩为1
        
       
        self.linear1 = nn.Linear(self.embed_dim, self.embed_dim)
        self.residual_linear = nn.Linear(self.embed_dim, self.embed_dim)
        
        
        self.multihead_attention = nn.MultiheadAttention(embed_dim=self.embed_dim, num_heads=self.num_heads)
        
       
        self.output_linear1 = nn.Linear(self.embed_dim, self.embed_dim // 2)
        self.output_linear2 = nn.Linear(self.embed_dim // 2, 1)

    def forward(self, x):
        # x shape: (batch_size, input_channels, input_width)
        
        x = self.conv1(x)
        x = torch.relu(x)
        x = self.conv2(x)
        x = torch.dropout(x,0.3,True)
        x = torch.relu(x)
        x = self.conv3(x)
        x = torch.relu(x)
        
        # Pooling to reduce sequence length to 1
        x = self.pool(x).squeeze(-1)  # Shape after pooling: (batch_size, embed_dim)
        
        x = torch.relu(self.linear1(x))
        residual = torch.relu(self.residual_linear(x))
        
        attn_input = x.unsqueeze(1).transpose(0, 1)  # Shape: (1, batch_size, embed_dim)
        
        attn_output, _ = self.multihead_attention(attn_input, attn_input, attn_input)
        
      
        attn_output = attn_output.transpose(0, 1).squeeze(1)  # Shape: (batch_size, embed_dim)
        
     
        attn_output += 0.5 * residual
        output =  torch.relu(self.output_linear1(attn_output))
        output= torch.dropout(output,0.3,True)
        output = self.output_linear2(output)
       
        return output

#### 3.初始化batch

In [ ]:

import torch
import random
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batchs = []
model = torch.load("../model/state-criric-model_new.pth").to('cpu')
for i in range(len(dataset)):
    once_prevStateJson,once_actions,once_nextStateJson = dataset[i]
    once_nextStateJson = copy.deepcopy(once_nextStateJson)
    once_prevStateJson = copy.deepcopy(once_prevStateJson)
    once_actions = copy.deepcopy(once_actions)
    if len(once_prevStateJson) != 5:continue
    if len(once_nextStateJson) != 5:continue
    x = []
    for serverId in range(len(once_prevStateJson)):
        prevStateJson = once_prevStateJson[serverId]
        nextStateJson = once_nextStateJson[serverId]
        actions = once_actions[serverId]
        if(prevStateJson is None or nextStateJson is None):continue  
        if('role' not in prevStateJson or 'role' not in nextStateJson):continue
        if(prevStateJson['role'] != 'state' or nextStateJson['role'] != 'state'):continue
        ens_rx_str = 'system_state.network.process.ens33:.RX Bytes'
        ens_tx_str = 'system_state.network.process.ens33:.TX Bytes'
        dis_ens_rx_bytes = abs(nextStateJson[ens_rx_str] - prevStateJson[ens_rx_str])
        dis_ens_tx_bytes = abs(nextStateJson[ens_tx_str] - prevStateJson[ens_tx_str])
        prevStateJson[ens_rx_str] = dis_ens_rx_bytes
        prevStateJson[ens_tx_str] = dis_ens_tx_bytes
   
        
        prevStateJson = datas.handleOriginFlattendJson(prevStateJson)
        # nextStateJson = datas.handleOriginFlattendJson(nextStateJson)
        x.append(prevStateJson)
    if(len(x) != 5)   :continue
    input_x = [list(ix.values()) for ix in x]
    input_x = torch.tensor([input_x],dtype=torch.float32)
    y_label =model(input_x).item()
    # for serverId in range(len(once_actions)):
    #     actions = once_actions[serverId]
    #     if('duration' not in  actions[0]): continue
    #     y_t = []
    #     for action in actions:
    #         y_t.append(action['duration'])
          
    #     if(len(y_t) == 0):continue
    #     y_label +=sum(y_t)/len(y_t)
    batchs.append((x,y_label))
    
    


In [ ]:
#system_state.memory.host.MemAvailable  system_state.network.process.ens33:.RX Bytes
server0 = {"cpu":[],"mem":[],"tx_bytes":[],"rx_bytes":[],"state_value":[]}
server1 = {"cpu":[],"mem":[],"tx_bytes":[],"rx_bytes":[],"state_value":[]}
server2 = {"cpu":[],"mem":[],"tx_bytes":[],"rx_bytes":[],"state_value":[]}
server3 = {"cpu":[],"mem":[],"tx_bytes":[],"rx_bytes":[],"state_value":[]}
server4 = {"cpu":[],"mem":[],"tx_bytes":[],"rx_bytes":[],"state_value":[]}
servers = [server0,server1,server2,server3,server4]
for xs ,y in batchs[20:]:
    for serverId in range(len(xs)):
        x = xs[serverId]
        servers[serverId]['cpu'].append(x['system_state.cpu.process.cpu'])
        servers[serverId]['mem'].append(x['system_state.memory.host.MemAvailable'])
        servers[serverId]['tx_bytes'].append(x['system_state.network.process.ens33:.TX Bytes'])
        servers[serverId]['rx_bytes'].append(x['system_state.network.process.ens33:.RX Bytes'])
        servers[serverId]['state_value'].append(y)


In [ ]:
def select_every_kth(input_list, k):
    # 检查k是否大于0，如果不是，则返回空列表
    if k <= 0:
        return []
    # 使用列表推导式每隔k个元素取值
    return [input_list[i] for i in range(0, len(input_list), k)]


def guiyihua(data):
    max_data = max(data)
    min_data = min(data)
    return [(x - min_data) / (max_data - min_data) for x in data]
from matplotlib import pyplot as plt
import numpy as np

In [ ]:
state_value_list = servers[0]['state_value']
state_value_list = [x  for x in state_value_list]
state_value_list = guiyihua(state_value_list)
k = 2500
colors = ['#008C8C','#E85827','#81D8D0','#800020','#F9DC24']
plt.bar(range(len(select_every_kth(state_value_list,k))),select_every_kth(state_value_list,k),color = '#002FA7',label='state_value',width=0.5, alpha=0.2)
tmp_list = []
for i,server in enumerate(servers):
    mem_list = server['mem']
    mem_list = guiyihua(mem_list)
    tmp_list.append(mem_list)
    plt.plot(select_every_kth(mem_list,k),label="host-" + str(i) + '-mem',color = colors[i],linewidth=0.6,linestyle='--',marker='^',markersize=3)
avg_mem_list = np.mean(tmp_list,axis=0)
plt.plot(select_every_kth(avg_mem_list,k),label="avg-mem",color = 'red',linewidth=0.9,linestyle='-',marker='o',markersize=4)
plt.legend(fontsize=8.5)
plt.grid(True)
plt.grid(linewidth=0.5,alpha = 0.7,linestyle='--')

plt.title('cluster(5 hosts) state value with memory available',fontsize=12)
#设置x轴标签
plt.xlabel('time step sampling',fontsize=12)
#设置y轴标签
plt.ylabel('corresponding value',fontsize=12)
dpis = [300,500,800,1200]
for dpi in dpis:
    plt.savefig(f'../out/memory_available-{dpi}.png',dpi = dpi)

In [ ]:
state_value_list = servers[0]['state_value']
state_value_list = guiyihua(state_value_list)
colors = ['#008C8C','#E85827','#81D8D0','#800020','#F9DC24']
k = 2500
plt.bar(range(len(select_every_kth(state_value_list,k))),select_every_kth(state_value_list,k),color = '#002FA7',label='state_value',width=0.5, alpha=0.2)
# plt.plot(select_every_kth(state_value_list,2500),label='state_value',linewidth=1.0,linestyle='-.',marker='o',markersize=4)
tmp_list =[]
for i,server in enumerate(servers):
    data_list = server['cpu']
    data_list = guiyihua(data_list)
    tmp_list.append(data_list)
    plt.plot(select_every_kth(data_list,k),label="host-" + str(i) + '-cpu',color = colors[i],linewidth=0.6,linestyle='--',marker='^',markersize=3)
avg_list = np.mean(tmp_list,axis=0)
plt.plot(select_every_kth(avg_list,k),label="avg-cpu",color = '#FF0000',linewidth=0.8,linestyle='-',marker='o',markersize=4)

plt.legend(fontsize=8.5)
plt.grid(True)
plt.grid(linewidth=0.5,alpha = 0.7,linestyle='--')

plt.title('cluster(5 hosts) state value with cpu usage',fontsize=12)
#设置x轴标签
plt.xlabel('time step sampling',fontsize=12)
#设置y轴标签
plt.ylabel('corresponding value',fontsize=12)
dpis = [300,500,800,1200]
for dpi in dpis:
    plt.savefig(f'../out/cpu_usage-{dpi}.png',dpi = dpi)


In [ ]:
state_value_list = servers[0]['state_value']
state_value_list = guiyihua(state_value_list)
colors = ['#008C8C','#E85827','#81D8D0','#800020','#F9DC24']
k = 2500
plt.bar(range(len(select_every_kth(state_value_list,k))),select_every_kth(state_value_list,k),color = '#8F4B28',label='state_value',width=0.5, alpha=0.2)
# plt.plot(select_every_kth(state_value_list,2500),label='state_value',linewidth=1.0,linestyle='-.',marker='o',markersize=4)
type_ = 'tx_bytes'
tmp_list = []
for i,server in enumerate(servers):
    data_list = server[type_]
    data_list = guiyihua(data_list)
    tmp_list.append(data_list)
    plt.plot(select_every_kth(data_list,k),label="host-" + str(i) + f'-{type_}',color = colors[i],linewidth=0.6,linestyle='--',marker='^',markersize=3)

avg_list = np.mean(tmp_list,axis=0)
plt.plot(range(len(select_every_kth(avg_list,k))),select_every_kth(avg_list,k),color = 'red',label='avg',linewidth=1.0,linestyle='-',marker='o',markersize=4)
plt.legend(fontsize=8.5)
plt.grid(True)
plt.grid(linewidth=0.5,alpha = 0.7,linestyle='--')

plt.title('cluster(5 hosts) state value with NetWork Send Bytes',fontsize=12)
#设置x轴标签
plt.xlabel('time step sampling',fontsize=12)
#设置y轴标签
plt.ylabel('corresponding value',fontsize=12)
dpis = [300,500,800,1200]
for dpi in dpis:
    plt.savefig(f'../out/{type_}-{dpi}.png',dpi = dpi)


In [ ]:
import numpy as np
state_value_list = servers[0]['state_value']
state_value_list = guiyihua(state_value_list)
colors = ['#008C8C','#E85827','#81D8D0','#800020','#F9DC24']
k = 2500
plt.bar(range(len(select_every_kth(state_value_list,k))),select_every_kth(state_value_list,k),color = '#8F4B28',label='state_value',width=0.5, alpha=0.2)
# plt.plot(select_every_kth(state_value_list,2500),label='state_value',linewidth=1.0,linestyle='-.',marker='o',markersize=4)
type_ = 'rx_bytes'
tmp_list = []
for i,server in enumerate(servers):
    data_list = server[type_]
    data_list = guiyihua(data_list)
    data_list =  [ x  for x in data_list]
    tmp_list.append(data_list)
    plt.plot(select_every_kth(data_list,k),label="host-" + str(i) + f'-{type_}',color = colors[i],linewidth=0.6,linestyle='--',marker='^',markersize=3)
avg_list = np.mean(tmp_list,axis=0)
plt.plot(range(len(select_every_kth(avg_list,k))),select_every_kth(avg_list,k),color = 'red',label='avg',linewidth=1.0,linestyle='-',marker='o',markersize=4)
plt.legend(fontsize=8.5)
plt.grid(True)
plt.grid(linewidth=0.5,alpha = 0.7,linestyle='--')

plt.title('cluster(5 hosts) state value with NetWork Receive Bytes',fontsize=12)
#设置x轴标签
plt.xlabel('time step sampling',fontsize=12)
#设置y轴标签
plt.ylabel('corresponding value',fontsize=12)
dpis = [300,500,800,1200]
for dpi in dpis:
    plt.savefig(f'../out/{type_}-{dpi}.png',dpi = dpi)
